# 3. `State Reducer`

## 3.1  什么是`State Reducer`

`State Reducer`是 LangGraph 中用于合并状态更新的核心机制。在 LangGraph 的 `StateGrph` 中，每个节点可以读取和写入共享状态，而 `Reducer` 定义了如何将多个节点对同一状态键的更新合并

`Reducer`的核心特征：
- 函数签名：`(Value, Value) ->  Value`，接受当前值和更新至，返回合并后的值
- 注解定义：通过 `Annotated[Type, reducer_function]` 为状态键指定 `Reducer`
- 默认行为：未指定 `Reducer` 的状态键使用覆盖策略（`Last-Write-Wins`）

## 3.2 如何定义`Reducer`

### 3.2.1 定义`Reducer`函数
`Reducer`本质上是一个二元合并函数，用于定义当同一个字段产生多个更新值时，LangGraph 应该如何将这些值合并为一个最终结果。

函数签名：`(Value, Value) -> Value`

In [1]:
def my_reducer(left: list[str], right: list[str]) -> list[str]:
    return left + right

left = ["a", "b"]
right = ["c"]
print(my_reducer(left, right))

['a', 'b', 'c']


其中，`my_reducer` 用于处理 `list[str]` 类型的数据。它接受两个参数列表：
- `left`：当前已累计的状态值
- `right`：本次待合并的新值
函数内部通过 `left + right` 将两个列表合并，并返回合并后的结果。

因此，该 `Reducer` 的作用是：当某个状态字段存在多次列表更新时，将这些列表内容追加合并，而不是直接覆盖原值。

### 3.2.2 将`Reduce`和状态字段关联

在 LangGraph 中，`Reducer`通常通过 Python 的 `typing.Annotated` 与状态字段进行关联。

`Annotated[]` 是 Python 提供的一种类型注解扩展机制，用于在原始类型之外附加额外的元数据信息。需要注意的是，`Annotated[]`本身不规定这些元数据的具体含义，它只负责在类型注解中保留这些信息。

严格来说，`Annotated`的第一个参数是被注解的原始类型，后续参数是附加的元数据。至于这些元数据表示什么、如何解析，则由使用它的框架或工具自行决定。

在 LangGraph 中，框架利用这一机制，将状态字段的类型和 `Reducer` 规则同时声明在字段定义中。其基本形式如下：

```python
Annotated[Type, reducer_function]
```

其中：
- `Type`：表示状态字段的数据类型
- `reducer_function`：表示该字段对应的`Reducer`函数
如：

```python
from typing import TypedDict, Annotated

class OverAllState(TypedDict):
    logs: Annotated[list[str], my_reducer]
    cur_id: str
```

### 3.2.3 常用的内置 `Reducer` 函数

1. `operator.add`
    是`Python`内置的加法操作函数，底层由 C 实现，它接受两个参数，等价于 `a（第一个参数） + b（第二个参数）`

2. `add_messages`
    是 LangGraph 中专用与合并消息列表的 `Reducer` 函数，常用于维护对话历史类的状态字段。函数签名如下：
   ```python
   def add_messages(
        left: Messages,
        right: Messages,
        *,
        format: Literal["langchain-openai"] | None = None,
    ) -> Messages:
        ...
        return merged
   ```

   参数说明：
    - `left`：状态中已有的消息列表；
    - `right`：当前节点返回的消息更新值；
    - `format`：可选参数，用于指定返回消息的格式，通常无需手动设置。
    - `left` 与 `right` 的类型均为 `Messages`。`Messages` 可以理解为 LangChain 消息对象的列表，其中每个元素都是 `BaseMessage` 或其子类的实例，

    常见子类包括：
    - HumanMessage：用户的输入消息；
    - AIMessage：AI 的回复消息；
    - SystemMessage：系统提示消息；
    - ToolMessage：工具调用的结果消息。
    - add_messages 处理的是对话消息序列，而非普通的字符串列表。

    `BaseMessage` 包含一个可选的 `id` 属性，用于唯一标识一条消息。`add_messages` 在合并 `left` 与 `right` 时，不是简单地执行列表拼接，而是依据消息的 `id` 进行合并：

    - 若 `right` 中的某条消息的 `id` 在 `left` 中不存在，则将该消息追加到结果列表末尾；
    - 若 `right` 中的某条消息的 `id` 与 `left` 中已有消息的 `id` 相同，则使用 `right` 中的新消息替换 `left` 中的旧消息。

    因此，`add_messages` 的作用可以概括为：**在保留历史消息的基础上追加新消息，并允许通过相同的消息 id 覆盖已有消息。**

    需要特别说明，add_messages 并非简单地对 left 与 right 求“并集”。更准确地说，它是一个基于消息 id 的消息列表合并函数：既支持追加新消息，也支持更新已有消息。

    可以理解为：
   ```python
   merged = left + right
   ```

    但若 `right` 中存在与 `left` 相同 `id` 的消息，则最终结果中不会出现重复消息，而是用 `right` 中的消息覆盖 `left` 中对应的旧消息。

    并不会应为子类产生其他操作，仅根据 `id` 进行新增或替换。

In [4]:
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage, AIMessage, SystemMessage

left = [
    SystemMessage(content="你是个善解人意的助手", id='1'),
    HumanMessage(content="你好", id='2'),
    AIMessage(content="你好~", id='3'),
]

right = [
    HumanMessage(content="我是老王，你是小王", id='2'),
    AIMessage(content="好的，我记住啦", id='3'),
    HumanMessage(content="你是谁？", id='4'),
    AIMessage(content="我是小王", id='5'),
]

merged = add_messages(left, right)

for msg in merged:
    print(msg)

content='我是小王' additional_kwargs={} response_metadata={} id='1' tool_calls=[] invalid_tool_calls=[]
content='我是老王，你是小王' additional_kwargs={} response_metadata={} id='2'
content='好的，我记住啦' additional_kwargs={} response_metadata={} id='3' tool_calls=[] invalid_tool_calls=[]
content='你是谁？' additional_kwargs={} response_metadata={} id='4'


## 3.3. 默认行为
如果某个 State 字段没有显式定义 `Reducer`，LangGraph 会使用默认的状态更新行为：后一次更新值会覆盖该字段原有的状态值。

换句话说，当节点返回的更新结果中包含某个字段时，如果该字段没有配置 `Reducer`，LangGraph 不会对新旧值进行追加、合并或累加，而是**直接使用本次返回的新值替换原来的旧值**。

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class OverAllState(TypedDict):
    logs: list[str]
    id: str

def node_a(state: OverAllState):
    return {
        "logs": ["node_a"],
        "id": "node_a"
    }

def node_b(state: OverAllState):
    return {
        "logs": ["node_b"],
        "id": "node_b"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'logs': ['node_b'], 'id': 'node_b'}
